In [ ]:
import subprocess

import pandas as pd
from vero.evaluator import run_evaluation  # noqa: F401

from vero_benchmarking.tasks import ALL_TASKS

task = ALL_TASKS["simpleqa"]
head_commit = subprocess.run(
    ["git", "rev-parse", "main"],
    cwd=task.project_path, capture_output=True, text=True, check=True,
).stdout.strip()

run_evaluation_kwargs = dict(
    project_path=str(task.project_path),
    dataset=str(task.dataset_path),
    split="test",
    commit=head_commit,
    task=str(task.task),
)


def compute_metrics(df: pd.DataFrame) -> dict[str, float]:
    correct = df["feedback"] == "A"
    incorrect = df["feedback"] == "B"
    max_turns_exceeded = df["feedback"].str.contains("turns")
    accuracy = correct.mean()
    accuracy_given_attempted = correct.sum() / (correct.sum() + incorrect.sum())
    f1_score = 2 * (accuracy * accuracy_given_attempted) / (accuracy + accuracy_given_attempted)
    return {
        "accuracy": float(accuracy),
        "accuracy_given_attempted": float(accuracy_given_attempted),
        "f1_score": float(f1_score),
        "num_max_turns_exceeded": int(max_turns_exceeded.sum()),
        "num_samples": int(len(df)),
    }

In [ ]:
results = {}

In [ ]:
for model in [
    "anthropic/claude-sonnet-4-5-20250929",
    "openai/gpt-4.1-mini-2025-04-14",
    "openai/gpt-4.1-2025-04-14",
]:
    if model in results:
        continue
    results[model] = await run_evaluation(  # noqa: F704
        task_params={"model": model}, **run_evaluation_kwargs
    )

In [ ]:
for model, result in results.items():
    df = result.sample_results_df()
    metrics = compute_metrics(df)
    print(f"Model: {model}")
    print(metrics)
    print("\n")